# Dataset Preprocessing — `merged_set.csv`

This notebook reproduces the full preprocessing pipeline applied to the merged GLD-v2 dataset before training and evaluation.

**Pipeline overview:**
1. Load & inspect the raw merged set
2. Score filtering (threshold ≥ 0.08)
3. Label analysis — identify and drop sparse labels
4. Sparse cell removal (region × label cells with < 100 samples)
5. Per-cell capping (max 400 samples per cell)
6. Stratified 80:20 split by (label, country)
7. Save outputs

In [ ]:
import numpy as np
import polars as pl

DATA_PATH     = "/mnt/yokoyamalab-nas/gldv2-full/merged_set.csv"
OUT_TRAIN     = "/mnt/yokoyamalab-nas/gldv2-full/merged_set_train.csv"
OUT_INDEX     = "/mnt/yokoyamalab-nas/gldv2-full/merged_set_index.csv"

SCORE_THRESHOLD = 0.08
CAP             = 400
MIN_CELL        = 100
TEST_SIZE       = 0.2
MIN_GROUP_SIZE  = 5
SEED            = 42

## 1. Load & inspect the raw merged set

In [ ]:
df_raw = pl.read_csv(DATA_PATH)
print(f"Shape : {df_raw.shape}")
print(f"Columns: {df_raw.columns}")
df_raw.head(5)

In [ ]:
# Source folder distribution (train vs index images)
df_raw["folder"].value_counts().sort("folder")

In [ ]:
# All 12 original labels and their raw counts
df_raw.group_by("predicted_label").len().sort("len", descending=True)

In [ ]:
# Score distribution overview
df_raw.select("predicted_score").describe()

In [ ]:
# How many rows are negative / below threshold?
below = df_raw.filter(pl.col("predicted_score") < SCORE_THRESHOLD)
print(f"Rows below {SCORE_THRESHOLD}: {len(below):,}  ({100*len(below)/len(df_raw):.1f}%)")

## 2. Score filtering

Keep only rows where the SigLIP classification confidence meets the minimum threshold.
Negative scores are cosine similarities below the decision boundary and are discarded.

In [ ]:
df = df_raw.filter(
    (pl.col("predicted_score") >= SCORE_THRESHOLD) &
    pl.col("region").is_not_null()
)
print(f"After score filter + null-region drop: {len(df):,}  (removed {len(df_raw)-len(df):,} rows)")

In [ ]:
# Remaining label counts after score filter
df.group_by("predicted_label").len().sort("len", descending=True)

## 3. Label analysis — identify sparse labels

A label is only useful for geo-balanced training if it has enough samples across **multiple geographic regions**.
We inspect how many of the 6 regions each label covers when applying the `MIN_CELL = 100` threshold.

In [ ]:
ALL_LABELS = sorted(df["predicted_label"].unique().to_list())
REGIONS    = sorted(df["region"].unique().to_list())
print(f"Labels ({len(ALL_LABELS)}): {ALL_LABELS}")
print(f"Regions ({len(REGIONS)}): {REGIONS}")

In [ ]:
# Full (region x label) cell size matrix
cell_matrix = (
    df.group_by(["predicted_label", "region"])
      .len()
      .pivot(on="region", index="predicted_label", values="len")
      .sort("predicted_label")
)
cell_matrix

In [ ]:
# Count how many regions each label has >= MIN_CELL samples
cell_counts = df.group_by(["predicted_label", "region"]).len()

coverage = (
    cell_counts
    .group_by("predicted_label")
    .agg(
        pl.len().alias("total_regions"),
        (pl.col("len") >= MIN_CELL).sum().alias("valid_regions"),
    )
    .sort("valid_regions", descending=True)
)
print(f"Min-cell threshold: {MIN_CELL}")
coverage

In [ ]:
# Labels with fewer than 5 valid regions are dropped
DROPPED_LABELS = coverage.filter(pl.col("valid_regions") < 5)["predicted_label"].to_list()
KEPT_LABELS    = coverage.filter(pl.col("valid_regions") >= 5)["predicted_label"].to_list()

print(f"Dropped ({len(DROPPED_LABELS)}): {sorted(DROPPED_LABELS)}")
print(f"Kept   ({len(KEPT_LABELS)}):    {sorted(KEPT_LABELS)}")

**Dropped labels:**
- `religious_other` — only **1/5** regions met the 100-sample threshold (Asia only)
- `other_heritage`  — only **3/6** regions
- `natural_landmark` — only **4/6** regions

Retaining these would create a severely imbalanced label with limited geographic diversity, undermining the goal of geo-balanced training.

In [ ]:
# Apply label filter
df = df.filter(pl.col("predicted_label").is_in(KEPT_LABELS))
print(f"After label filter: {len(df):,} rows, {df['predicted_label'].n_unique()} labels")

## 4. Sparse cell removal

Even within the 9 kept labels, some (region × label) cells are too small to be useful.
Cells with fewer than `MIN_CELL = 100` samples are dropped entirely.

In [ ]:
cell_sizes = (
    df.group_by(["region", "predicted_label"])
      .len()
      .rename({"len": "cell_size"})
)

sparse_cells = cell_sizes.filter(pl.col("cell_size") < MIN_CELL).sort("cell_size")
print(f"Sparse cells to drop ({len(sparse_cells)}):")
sparse_cells

In [ ]:
df = (
    df.join(cell_sizes, on=["region", "predicted_label"], how="left")
      .filter(pl.col("cell_size") >= MIN_CELL)
      .drop("cell_size")
)
print(f"After sparse cell removal: {len(df):,} rows")

## 5. Per-cell capping

To counteract the dominance of densely populated regions (particularly Europe),
each (region × label) cell is capped at `CAP = 400` samples via random sampling.

In [ ]:
# Distribution before capping
before_cap = (
    df.group_by(["region", "predicted_label"])
      .len()
      .sort(["predicted_label", "region"])
)
print(f"Cells above cap ({CAP}): {before_cap.filter(pl.col('len') > CAP).height}")
before_cap

In [ ]:
rng = np.random.default_rng(SEED)
df = df.with_columns(pl.Series("__rand", rng.random(len(df))))
df = df.with_columns(
    pl.col("__rand").rank("ordinal").over(["region", "predicted_label"]).alias("__rank")
)
df = df.filter(pl.col("__rank") <= CAP).drop(["__rand", "__rank"])

print(f"After capping at {CAP}: {len(df):,} rows")

In [ ]:
# Distribution after capping
after_cap = (
    df.group_by(["region", "predicted_label"])
      .len()
      .sort(["predicted_label", "region"])
)
after_cap

In [ ]:
# Per-label total after capping
df.group_by("predicted_label").len().sort("predicted_label")

## 6. Stratified 80:20 split

The balanced pool is split into **training** (80%) and **index** (20%) sets.
Stratification is done by **(label, country)** to preserve the geographic distribution in both splits.

Groups with fewer than `MIN_GROUP_SIZE = 5` images are assigned entirely to training
to avoid index-only singletons.

In [ ]:
# How many (label, country) groups exist and their size distribution?
groups = df.group_by(["predicted_label", "country"]).len().sort("len", descending=True)
print(f"Total (label, country) groups: {len(groups)}")
print(f"Groups below MIN_GROUP_SIZE ({MIN_GROUP_SIZE}): {groups.filter(pl.col('len') < MIN_GROUP_SIZE).height}")
groups.describe()

In [ ]:
rng2 = np.random.default_rng(SEED + 1)
df = df.with_columns(pl.Series("__rand", rng2.random(len(df))))
df = df.with_columns([
    pl.col("__rand").rank("ordinal").over(["predicted_label", "country"]).alias("__rank"),
    pl.len().over(["predicted_label", "country"]).alias("__group_size"),
])
df = df.with_columns(
    (
        (pl.col("__rank") <= (pl.col("__group_size") * TEST_SIZE).floor()) &
        (pl.col("__group_size") >= MIN_GROUP_SIZE)
    ).alias("__is_index")
)

train_df = df.filter(~pl.col("__is_index")).drop(["__rand", "__rank", "__group_size", "__is_index"])
index_df = df.filter( pl.col("__is_index")).drop(["__rand", "__rank", "__group_size", "__is_index"])

total = len(train_df) + len(index_df)
print(f"Train : {len(train_df):,}  ({100*len(train_df)/total:.1f}%)")
print(f"Index : {len(index_df):,}  ({100*len(index_df)/total:.1f}%)")

In [ ]:
# Per-label split breakdown
train_df.group_by("predicted_label").len().rename({"len": "train"}).join(
    index_df.group_by("predicted_label").len().rename({"len": "index"}),
    on="predicted_label"
).with_columns(
    (pl.col("index") / (pl.col("train") + pl.col("index"))).round(3).alias("index_ratio")
).sort("predicted_label")

In [ ]:
# Source folder distribution across splits
print("Train folder distribution:")
print(train_df["folder"].value_counts().sort("folder"))
print("\nIndex folder distribution:")
print(index_df["folder"].value_counts().sort("folder"))

In [ ]:
# Region distribution across splits
train_df.group_by("region").len().rename({"len": "train"}).join(
    index_df.group_by("region").len().rename({"len": "index"}),
    on="region"
).sort("region")

## 7. Save outputs

In [ ]:
train_df.write_csv(OUT_TRAIN)
index_df.write_csv(OUT_INDEX)
print(f"Saved: {OUT_TRAIN}  ({len(train_df):,} rows)")
print(f"Saved: {OUT_INDEX}  ({len(index_df):,} rows)")